# مسئلهٔ ۱ — V2 / مدل A3: ResNet18 + Temporal Attention Pooling

این مدل برای feature هر یک از ۱۶ فریم یک score یاد می‌گیرد، scoreها را با softmax به attention weight تبدیل می‌کند و feature ویدئو را به‌شکل weighted sum می‌سازد. برخلاف mean-max، انتخاب فریم‌ها یادگرفتنی و نرم است.

ResNet18 همچنان freeze است؛ بنابراین در این ablation فقط temporal aggregator نسبت به A1 و A2 تغییر می‌کند. attention weightها ذخیره می‌شوند تا در تحلیل خطا بدانیم مدل روی کدام بخش پنجره تمرکز کرده است؛ این وزن‌ها توضیح علّی قطعی نیستند.

In [1]:
from __future__ import annotations

from pathlib import Path
import json
import random

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    ConfusionMatrixDisplay, accuracy_score, average_precision_score,
    confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score,
)
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

DATA_ROOT = Path(r'P:\NexarCollisionData')
SEQUENCE_MANIFEST_PATH = DATA_ROOT / 'sequence_manifest_v2.csv'
FEATURE_CACHE_PATH = DATA_ROOT / 'processed_v2' / 'resnet18_imagenet_features_v2_w2_16x224x320.pt'
MODEL_DIR = DATA_ROOT / 'models_v2'
MODEL_NAME = 'resnet18_temporal_attention_frozen'

NUM_FRAMES = 16
FEATURE_DIM = 512
ATTENTION_HIDDEN_DIM = 128
BATCH_SIZE = 64
EPOCHS = 40
PATIENCE = 8
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
SEED = 42

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert SEQUENCE_MANIFEST_PATH.exists(), 'Run notebook 08 first.'
assert FEATURE_CACHE_PATH.exists(), 'Run notebook 10 first to create the feature cache.'
print(f'Device: {device}')

Device: cpu


In [2]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
sequence_manifest = pd.read_csv(SEQUENCE_MANIFEST_PATH).copy()
sequence_manifest['video_id'] = sequence_manifest['video_id'].astype(str)
sequence_manifest['label'] = sequence_manifest['label'].astype(int)
sequence_manifest = sequence_manifest.sort_values('video_id', key=lambda series: series.astype(int)).reset_index(drop=True)

feature_payload = torch.load(FEATURE_CACHE_PATH, map_location='cpu', weights_only=False)
features = feature_payload['features'].float()
labels = feature_payload['labels'].long()
assert features.shape == (600, NUM_FRAMES, FEATURE_DIM)
assert labels.tolist() == sequence_manifest['label'].tolist()
assert feature_payload['sequence_ids'] == sequence_manifest['sequence_id'].tolist()

train_indices = np.flatnonzero(sequence_manifest['split'].eq('train').to_numpy())
validation_indices = np.flatnonzero(sequence_manifest['split'].eq('validation').to_numpy())
assert len(train_indices) == 480 and len(validation_indices) == 120
display(pd.crosstab(sequence_manifest['split'], sequence_manifest['label']))

label,0,1
split,,
train,240,240
validation,60,60


In [3]:
class SequenceFeatureDataset(Dataset):
    def __init__(self, features: torch.Tensor, labels: torch.Tensor, indices: np.ndarray):
        self.features = features
        self.labels = labels
        self.indices = torch.as_tensor(indices, dtype=torch.long)

    def __len__(self) -> int:
        return len(self.indices)

    def __getitem__(self, index: int):
        source_index = self.indices[index]
        return self.features[source_index], self.labels[source_index], int(source_index)

class ResNet18TemporalAttentionHead(nn.Module):
    def __init__(self, feature_dim: int = FEATURE_DIM, attention_hidden_dim: int = ATTENTION_HIDDEN_DIM, dropout: float = 0.35):
        super().__init__()
        self.attention_scorer = nn.Sequential(
            nn.Linear(feature_dim, attention_hidden_dim),
            nn.Tanh(),
            nn.Linear(attention_hidden_dim, 1),
        )
        self.classifier = nn.Sequential(
            nn.LayerNorm(feature_dim),
            nn.Dropout(dropout),
            nn.Linear(feature_dim, 1),
        )

    def forward(self, sequence_features: torch.Tensor, frame_mask: torch.Tensor | None = None, return_attention: bool = False):
        attention_logits = self.attention_scorer(sequence_features).squeeze(-1)
        if frame_mask is not None:
            attention_logits = attention_logits.masked_fill(~frame_mask.bool(), float('-inf'))
        attention_weights = torch.softmax(attention_logits, dim=1)
        video_features = torch.sum(sequence_features * attention_weights.unsqueeze(-1), dim=1)
        logits = self.classifier(video_features).squeeze(1)
        return (logits, attention_weights) if return_attention else logits

train_loader = DataLoader(SequenceFeatureDataset(features, labels, train_indices), batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
validation_loader = DataLoader(SequenceFeatureDataset(features, labels, validation_indices), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
model = ResNet18TemporalAttentionHead().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
criterion = nn.BCEWithLogitsLoss()
print(model)

ResNet18TemporalAttentionHead(
  (attention_scorer): Sequential(
    (0): Linear(in_features=512, out_features=128, bias=True)
    (1): Tanh()
    (2): Linear(in_features=128, out_features=1, bias=True)
  )
  (classifier): Sequential(
    (0): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
    (1): Dropout(p=0.35, inplace=False)
    (2): Linear(in_features=512, out_features=1, bias=True)
  )
)


In [4]:
def binary_metrics(y_true: np.ndarray, probabilities: np.ndarray, threshold: float) -> dict:
    predictions = (probabilities >= threshold).astype(int)
    return {
        'threshold': float(threshold),
        'accuracy': float(accuracy_score(y_true, predictions)),
        'precision': float(precision_score(y_true, predictions, zero_division=0)),
        'recall': float(recall_score(y_true, predictions, zero_division=0)),
        'f1': float(f1_score(y_true, predictions, zero_division=0)),
        'roc_auc': float(roc_auc_score(y_true, probabilities)),
        'pr_auc': float(average_precision_score(y_true, probabilities)),
        'confusion_matrix': confusion_matrix(y_true, predictions).tolist(),
    }

def evaluate(model: nn.Module, loader: DataLoader, include_attention: bool = False):
    model.eval()
    labels_out, probabilities_out, indices_out, attention_out = [], [], [], []
    with torch.inference_mode():
        for batch_features, batch_labels, batch_indices in loader:
            batch_features = batch_features.to(device)
            if include_attention:
                logits, weights = model(batch_features, return_attention=True)
                attention_out.append(weights.cpu().numpy())
            else:
                logits = model(batch_features)
            labels_out.append(batch_labels.numpy())
            probabilities_out.append(torch.sigmoid(logits).cpu().numpy())
            indices_out.append(batch_indices.numpy())
    outputs = (np.concatenate(labels_out), np.concatenate(probabilities_out), np.concatenate(indices_out))
    return (*outputs, np.concatenate(attention_out)) if include_attention else outputs

best_pr_auc = -np.inf
best_epoch = 0
epochs_without_improvement = 0
history = []
best_model_path = MODEL_DIR / f'{MODEL_NAME}_best.pt'

for epoch in range(1, EPOCHS + 1):
    model.train()
    loss_sum = 0.0
    for batch_features, batch_labels, _ in train_loader:
        batch_features = batch_features.to(device)
        batch_labels = batch_labels.float().to(device)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(batch_features), batch_labels)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * len(batch_labels)

    validation_labels, validation_probabilities, _ = evaluate(model, validation_loader)
    metrics_at_05 = binary_metrics(validation_labels, validation_probabilities, 0.5)
    record = {
        'epoch': epoch,
        'train_loss': loss_sum / len(train_loader.dataset),
        'validation_accuracy_at_0_5': metrics_at_05['accuracy'],
        'validation_f1_at_0_5': metrics_at_05['f1'],
        'validation_recall_at_0_5': metrics_at_05['recall'],
        'validation_pr_auc': metrics_at_05['pr_auc'],
        'validation_roc_auc': metrics_at_05['roc_auc'],
    }
    history.append(record)
    print(record)

    if record['validation_pr_auc'] > best_pr_auc:
        best_pr_auc = record['validation_pr_auc']
        best_epoch = epoch
        epochs_without_improvement = 0
        torch.save({
            'model_state_dict': model.state_dict(), 'epoch': epoch,
            'validation_pr_auc': best_pr_auc, 'model_name': MODEL_NAME,
            'feature_dim': FEATURE_DIM, 'attention_hidden_dim': ATTENTION_HIDDEN_DIM,
            'num_frames': NUM_FRAMES, 'encoder': 'ResNet18_Weights.IMAGENET1K_V1 (frozen)',
        }, best_model_path)
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print(f'Early stopping at epoch {epoch}; best epoch: {best_epoch}')
            break

history_path = MODEL_DIR / f'{MODEL_NAME}_training_history.csv'
pd.DataFrame(history).to_csv(history_path, index=False)
print(f'Best epoch by validation PR-AUC: {best_epoch}, PR-AUC={best_pr_auc:.4f}')

{'epoch': 1, 'train_loss': 0.7563048283259074, 'validation_accuracy_at_0_5': 0.5333333333333333, 'validation_f1_at_0_5': 0.6626506024096386, 'validation_recall_at_0_5': 0.9166666666666666, 'validation_pr_auc': 0.5948592372597181, 'validation_roc_auc': 0.6147222222222222}
{'epoch': 2, 'train_loss': 0.6872886101404826, 'validation_accuracy_at_0_5': 0.6416666666666667, 'validation_f1_at_0_5': 0.6260869565217392, 'validation_recall_at_0_5': 0.6, 'validation_pr_auc': 0.6605580766179004, 'validation_roc_auc': 0.7136111111111111}
{'epoch': 3, 'train_loss': 0.6452939013640085, 'validation_accuracy_at_0_5': 0.6583333333333333, 'validation_f1_at_0_5': 0.6095238095238096, 'validation_recall_at_0_5': 0.5333333333333333, 'validation_pr_auc': 0.6879518118612655, 'validation_roc_auc': 0.7463888888888889}
{'epoch': 4, 'train_loss': 0.5961026191711426, 'validation_accuracy_at_0_5': 0.7333333333333333, 'validation_f1_at_0_5': 0.7377049180327869, 'validation_recall_at_0_5': 0.75, 'validation_pr_auc': 0.7

In [5]:
checkpoint = torch.load(best_model_path, map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
validation_labels, validation_probabilities, validation_indices_out, validation_attention = evaluate(
    model, validation_loader, include_attention=True
)

threshold_table = pd.DataFrame([
    binary_metrics(validation_labels, validation_probabilities, float(threshold))
    for threshold in np.round(np.arange(0.10, 0.901, 0.01), 2)
])
selected_row = threshold_table.sort_values(['f1', 'recall', 'precision'], ascending=False).iloc[0]
selected_threshold = float(selected_row['threshold'])
metrics_at_05 = binary_metrics(validation_labels, validation_probabilities, 0.5)
metrics_at_selected_threshold = binary_metrics(validation_labels, validation_probabilities, selected_threshold)

prediction_table = sequence_manifest.iloc[validation_indices_out].copy().reset_index(drop=True)
prediction_table['positive_probability'] = validation_probabilities
prediction_table['prediction_at_0_5'] = (validation_probabilities >= 0.5).astype(int)
prediction_table['prediction_at_selected_threshold'] = (validation_probabilities >= selected_threshold).astype(int)
prediction_table['selected_threshold'] = selected_threshold
for frame_index in range(NUM_FRAMES):
    prediction_table[f'attention_{frame_index:02d}'] = validation_attention[:, frame_index]
prediction_table['most_attended_frame_index'] = validation_attention.argmax(axis=1)
timestamp_columns = [f'timestamp_{frame_index:02d}' for frame_index in range(NUM_FRAMES)]
prediction_table['most_attended_timestamp'] = [
    row[timestamp_columns[int(frame_index)]]
    for (_, row), frame_index in zip(prediction_table.iterrows(), prediction_table['most_attended_frame_index'])
]

predictions_path = MODEL_DIR / f'{MODEL_NAME}_validation_predictions.csv'
threshold_path = MODEL_DIR / f'{MODEL_NAME}_threshold_curve.csv'
metrics_path = MODEL_DIR / f'{MODEL_NAME}_metrics.json'
attention_summary_path = MODEL_DIR / f'{MODEL_NAME}_attention_summary.csv'
prediction_table.to_csv(predictions_path, index=False)
threshold_table.to_csv(threshold_path, index=False)
prediction_table[['video_id', 'label', 'positive_probability', 'most_attended_frame_index', 'most_attended_timestamp']].to_csv(attention_summary_path, index=False)

metrics_payload = {
    'model_name': MODEL_NAME,
    'evaluation_scope': 'clip-level validation on fixed V2-W2 sequences; not full-MP4 sliding-window inference',
    'selection_metric': 'validation PR-AUC at the saved checkpoint',
    'best_epoch': int(checkpoint['epoch']),
    'selected_threshold_by_validation_f1': selected_threshold,
    'metrics_at_threshold_0_5': metrics_at_05,
    'metrics_at_selected_threshold': metrics_at_selected_threshold,
    'encoder': 'ResNet18 ImageNet frozen',
    'temporal_aggregator': 'additive temporal attention over 16 frame features',
    'input_shape_per_sequence': [NUM_FRAMES, 3, 224, 320],
}
metrics_path.write_text(json.dumps(metrics_payload, indent=2), encoding='utf-8')

figure, axes = plt.subplots(1, 2, figsize=(10, 4))
ConfusionMatrixDisplay.from_predictions(validation_labels, (validation_probabilities >= 0.5).astype(int), ax=axes[0], colorbar=False)
axes[0].set_title('Validation, threshold = 0.50')
ConfusionMatrixDisplay.from_predictions(validation_labels, (validation_probabilities >= selected_threshold).astype(int), ax=axes[1], colorbar=False)
axes[1].set_title(f'Validation, threshold = {selected_threshold:.2f}')
figure.tight_layout()
confusion_path = MODEL_DIR / f'{MODEL_NAME}_confusion_matrices.png'
figure.savefig(confusion_path, dpi=160)
plt.close(figure)

comparison_sources = [
    ('A1 ResNet18 + mean pooling', MODEL_DIR / 'resnet18_mean_pooling_frozen_metrics.json'),
    ('A2 ResNet18 + mean-max pooling', MODEL_DIR / 'resnet18_meanmax_pooling_frozen_metrics.json'),
]
comparison_rows = []
for name, path in comparison_sources:
    if path.exists():
        payload = json.loads(path.read_text(encoding='utf-8'))
        result = payload['metrics_at_selected_threshold']
        comparison_rows.append({
            'model': name, 'f1': result['f1'], 'recall': result['recall'],
            'precision': result['precision'], 'pr_auc': result['pr_auc'],
            'threshold': payload['selected_threshold_by_validation_f1'],
        })
comparison_rows.append({
    'model': 'A3 ResNet18 + temporal attention',
    'f1': metrics_at_selected_threshold['f1'], 'recall': metrics_at_selected_threshold['recall'],
    'precision': metrics_at_selected_threshold['precision'], 'pr_auc': metrics_at_selected_threshold['pr_auc'],
    'threshold': selected_threshold,
})
comparison_table = pd.DataFrame(comparison_rows)
comparison_path = MODEL_DIR / 'v2_temporal_pooling_ablation_comparison.csv'
comparison_table.to_csv(comparison_path, index=False)

print('Metrics at threshold 0.50:')
print(metrics_at_05)
print('Metrics at validation-selected threshold:')
print(metrics_at_selected_threshold)
display(comparison_table)
print(f'Model: {best_model_path}')
print(f'Attention summary: {attention_summary_path}')

Metrics at threshold 0.50:
{'threshold': 0.5, 'accuracy': 0.75, 'precision': 0.75, 'recall': 0.75, 'f1': 0.75, 'roc_auc': 0.7755555555555556, 'pr_auc': 0.716780443131305, 'confusion_matrix': [[45, 15], [15, 45]]}
Metrics at validation-selected threshold:
{'threshold': 0.39, 'accuracy': 0.7416666666666667, 'precision': 0.6883116883116883, 'recall': 0.8833333333333333, 'f1': 0.7737226277372263, 'roc_auc': 0.7755555555555556, 'pr_auc': 0.716780443131305, 'confusion_matrix': [[36, 24], [7, 53]]}


,model,f1,recall,precision,pr_auc,threshold
0,A1 ResNet18 + mean pooling,0.762712,0.750000,0.775862,0.727511,0.47
1,A2 ResNet18 + mean-max pooling,0.787402,0.833333,0.746269,0.742437,0.44
2,A3 ResNet18 + temporal attention,0.773723,0.883333,0.688312,0.716780,0.39


Model: P:\NexarCollisionData\models_v2\resnet18_temporal_attention_frozen_best.pt
Attention summary: P:\NexarCollisionData\models_v2\resnet18_temporal_attention_frozen_attention_summary.csv


## تصمیم پس از A3

اگر A3 از A2 در F1 و Recall بهتر نبود، attention به‌تنهایی ارزش پیچیدگی اضافه ندارد و A2 را نگه می‌داریم. سپس یا ResNet18 را با fine-tuning محدود و augmentation سازگار بهبود می‌دهیم، یا در صورت نیاز به ترتیب زمانی صریح به سراغ GRU + attention می‌رویم.